# Tutorial 08: Advanced Multi-Agent Workflows

##  Learning Objectives
After completing this notebook, you will be able to:
- Build **custom workflow graphs** with `WorkflowBuilder` for complex routing
- Create **CustomExecutors** with business logic
- Implement **fan-out/fan-in patterns** for parallel processing
- Use **AI-powered orchestration** with `MagenticBuilder`
- Handle **conditional routing** and complex workflow patterns
- Understand **when to use advanced patterns** compared to basic workflows

##  Prerequisites

Before starting this tutorial, you should have completed:
- **Tutorial 05: Multi-Agent Workflows** (SequentialBuilder, ConcurrentBuilder)

This tutorial builds on those concepts with more advanced patterns!

##  What Makes It "Advanced"?

**Basic Workflows (Tutorial 05):**
-  Simple, declarative patterns
-  Sequential or parallel execution
-  No custom logic needed
- Examples: `SequentialBuilder`, `ConcurrentBuilder`

**Advanced Workflows (This Tutorial):**
-  Custom business logic in CustomExecutors
-  Conditional routing and branching
-  Fan-out/fan-in with custom gates
-  AI-powered dynamic orchestration
- Examples: `WorkflowBuilder`, `MagenticBuilder`

### When to Use Advanced Patterns

| Use Case | Pattern | Reason |
|----------|---------|--------|
| Approval workflows | WorkflowBuilder | Need conditional routing based on approval status |
| Budget validation gates | WorkflowBuilder | Need validation before proceeding |
| Multi-stage pipelines | WorkflowBuilder | Need custom control flow |
| Dynamic task planning | MagenticBuilder | Optimal workflow not known in advance |
| Adaptive systems | MagenticBuilder | Need AI to decide coordination strategy |

---

## Step 1: Setup and Imports

In [ ]:
import asyncio
from typing import cast

from agent_framework import (
    # Advanced workflow builders
    WorkflowBuilder,
    # Workflow components
    Executor,
    AgentExecutor,
    handler,
    WorkflowContext,
    # Events for tracking
    WorkflowEvent,
    AgentResponseUpdate,
    # Base types
    Message,
)
from agent_framework.orchestrations import MagenticBuilder
from agent_framework.azure import AzureOpenAIChatClient
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential
from dotenv import load_dotenv

load_dotenv(override=True)
print(" Import successful!")
print("📦 Advanced workflow builders ready: WorkflowBuilder, MagenticBuilder")

In [ ]:
import os
mcp_server_uri = os.getenv("MCP_SERVER_URI")
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === Authentication Method Selection ===
# True: API Key authentication, False: DefaultAzureCredential authentication
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("🔑 Authentication method: API Key")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # The framework automatically reads the AZURE_OPENAI_API_KEY environment variable
    # and prioritizes it over credential, so we explicitly remove it
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 Authentication method: DefaultAzureCredential")

print(f"MCP Server URI: {mcp_server_uri}")
print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

## Setting Up a Tracer with OpenTelemetry
Using an OpenTelemetry tracer is useful for debugging multi-agent systems.
Since `setup_observability` is not provided in this environment's Agent Framework,
we manually set up an OpenTelemetry `TracerProvider` (including Exporter/Processor) and enable instrumentation with `enable_instrumentation()`.
Here we use OTLP gRPC (e.g., Jaeger / OpenTelemetry Collector on port `4317`) as the trace destination.

Jaeger UI: [http://localhost:16686](http://localhost:16686)

In [ ]:
import os

from agent_framework.observability import configure_otel_providers, get_tracer

service_name = "agent_framework"
otlp_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4317")

# Environment variable-based configuration (Agent Framework reads standard OTEL_* variables)
os.environ.setdefault("OTEL_SERVICE_NAME", service_name)
os.environ.setdefault("OTEL_EXPORTER_OTLP_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_PROTOCOL", "grpc")
os.environ.setdefault("ENABLE_INSTRUMENTATION", "true")
os.environ.setdefault("OTEL_METRICS_EXPORTER", "none")  # Disable Metrics (change as needed)

# Specify enable_sensitive_data=True to enable collection of sensitive data (OpenAI IN/OUT data)
configure_otel_providers(enable_sensitive_data=True)

tracer = get_tracer()
print(f"Observability configured: service={service_name} otlp={otlp_endpoint}")

In [ ]:
from agent_framework import WorkflowViz
from IPython.display import SVG, display
import os

def visualize_workflow(workflow, filename="workflow_diagram", show_preview=True):
    """
    Function to output workflow graph information, save as SVG, and display a preview
    
    Args:
        workflow: The workflow object to visualize
        filename: The filename to save (without extension)
        show_preview: Whether to display the preview
    
    Returns:
        The path of the saved SVG file
    """
    # Create WorkflowViz object
    viz = WorkflowViz(workflow)
    
    # 3. Save as SVG file
    try:
        svg_path = viz.export(format="svg", filename=filename)
        print("=" * 60)
        print(f"✅ SVG file saved: {svg_path}")
        print("=" * 60)
        print()
        
        # 4. Display SVG preview
        if show_preview and os.path.exists(svg_path):
            display(SVG(filename=svg_path))
        
        return svg_path
        
    except ImportError as e:
        print("❌ Error: 'graphviz' package is not installed")
        print("Installation: pip install agent-framework[viz] --pre")
        print(f"Details: {e}")
        return None
    except Exception as e:
        print(f"❌ Error occurred: {e}")
        return None

## Step 2: Create Specialist Agents

We'll reuse the travel agent specialists from Tutorial 05.

In [ ]:
async def create_travel_agents():
    """
    Creates specialist travel planning agents.
    """
    # Note: Using AzureCliCredential without context manager to avoid session termination issues
    # Credentials are reused across multiple agent calls
    # credential = AzureCliCredential()
    # chat_client = AzureAIAgentClient(async_credential=credential)
    # Create Azure OpenAI client
    chat_client = AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment,
    )
    # Flight specialist
    flight_agent = chat_client.as_agent(
        instructions="""
        You are a flight booking specialist.
        Provide concise and practical flight recommendations considering:
        - Best timing for booking
        - Airline preferences and quality
        - Connection strategies
        - Price vs. convenience tradeoffs
        
        Keep responses concise (max 2-3 sentences).
        """,
        name="FlightExpert",
    )
    
    # Hotel specialist
    hotel_agent = chat_client.as_agent(
        instructions="""
        You are a hotel booking specialist.
        Provide concise hotel recommendations considering:
        - Best areas for tourists
        - Value for money
        - Proximity to attractions and transportation
        - Hotel quality and amenities
        
        Keep responses concise (max 2-3 sentences).
        """,
        name="HotelExpert",
    )
    
    # Activities specialist
    activities_agent = chat_client.as_agent(
        instructions="""
        You are a local activities and experiences specialist.
        Provide concise activity recommendations considering:
        - Must-see attractions
        - Local hotspots and hidden gems
        - Cultural experiences
        - Dining and gourmet food
        
        Keep responses concise (max 2-3 sentences).
        """,
        name="ActivitiesExpert",
    )
    
    return flight_agent, hotel_agent, activities_agent

print(" Agent factory created")

## Step 3: CustomExecutor - WorkflowBuilder

**CustomExecutors** allow you to add business logic to workflows:
- Validation gates (check conditions before proceeding)
- Data transformations (modify messages between agents)
- Conditional routing (send to different agents based on logic)
- Aggregation logic (combine results in custom ways)

### Creating a CustomExecutor

Every CustomExecutor must:
1. **Inherit from `Executor`**
2. **Call `super().__init__(id="unique_id")` in `__init__`**
3. **Define a `@handler` method with `WorkflowContext` type annotation**
4. **Use `ctx.send_message()` to pass data** to downstream executors

In [ ]:
# CustomExecutor for budget validation
class BudgetValidator(Executor):
    """Validates travel budget before proceeding to planning agents."""
    
    def __init__(self):
        # Required: Call super().__init__() with a unique ID
        super().__init__(id="budget_validator")
    
    @handler
    async def validate_budget(self, request: str, ctx: WorkflowContext[str]):
        """
        Checks if budget is mentioned and forwards to downstream agents.
        
        This demonstrates:
        - Executing business logic (budget keyword detection)
        - Message transformation (adding context)
        - Passing to downstream executors via ctx.send_message()
        """
        print(f"    Budget validation: Analyzing request...")
        
        # Business logic: Check for budget-related keywords
        budget_keywords = ['budget', '$', 'price', 'cost', 'cheap', 'expensive', 'affordable']
        has_budget = any(word in request.lower() for word in budget_keywords)
        
        if has_budget:
            print(f"    Budget considerations detected")
            # Transform message with budget context
            enhanced_request = f"Budget-focused travel request: {request}"
        else:
            print(f"     No budget specified - assuming standard pricing")
            # Add budget reminder
            enhanced_request = f"Standard travel request (consider mid-range options): {request}"
        
        # Send to downstream agents (fan-out happens via edges)
        await ctx.send_message(enhanced_request)

print(" BudgetValidator executor created")

## Step 4: Building a Custom Workflow Graph

**WorkflowBuilder** gives you full control over the workflow graph:
- Manually add executors and edges
- Create fan-out patterns (1 → many)
- Create fan-in patterns (many → 1)
- Add conditional routing
- Build complex DAGs

### Graph Building Patterns

```python
workflow = (
    WorkflowBuilder()
    .set_start_executor(start_node)           # Define entry point
    .add_fan_out_edges(validator, [a, b, c])  # 1 → many
    .add_edge(a, aggregator)                   # Individual edges
    .add_edge(b, aggregator)
    .add_edge(c, aggregator)
    .build()                                   # Validate and create
)
```

In [ ]:

"""
Demonstrates WorkflowBuilder: Custom DAG with validation gate and fan-out/fan-in.

Workflow graph:
User input → Budget validation → Fan-out to 3 agents → Aggregation → Final output
"""
print("="*70)
print("Custom Workflow: Budget Validation + Parallel Agents + Aggregation")
print("="*70)
print("\nWorkflow Graph:")
print("  User → Budget Validator → Fan-out ┬→ Flight Expert")
print("                                    ├→ Hotel Expert     → Aggregation")
print("                                    └→ Activities Expert")
print()

# Import required types
from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
)
from typing import Never

# Create an aggregator that collects and formats all agent responses
class TravelPlanAggregator(Executor):
    """Collects responses from all travel agents and formats the final plan"""
    
    def __init__(self):
        super().__init__(id="travel_plan_aggregator")
    
    @handler
    async def aggregate(
        self, 
        results: list[AgentExecutorResponse], 
        ctx: WorkflowContext[Never, str]
    ) -> None:
        """Aggregates all agent responses into a formatted travel plan"""
        # Extract responses by agent (silently - no print here)
        responses_by_agent = {}
        for result in results:
            agent_id = result.executor_id
            response_text = result.agent_response.text if result.agent_response else ""
            if response_text:
                responses_by_agent[agent_id] = response_text
        
        # Format unified plan
        plan = "\n" + "="*70 + "\n"
        plan += "📋 Your Complete Budget Barcelona Travel Plan\n"
        plan += "="*70 + "\n\n"
        
        if "FlightExpert" in responses_by_agent:
            plan += "  Flights:\n"
            plan += "─"*70 + "\n"
            plan += f"{responses_by_agent['FlightExpert']}\n\n"
        
        if "HotelExpert" in responses_by_agent:
            plan += " Accommodation:\n"
            plan += "─"*70 + "\n"
            plan += f"{responses_by_agent['HotelExpert']}\n\n"
        
        if "ActivitiesExpert" in responses_by_agent:
            plan += "🎭 Activities & Experiences:\n"
            plan += "─"*70 + "\n"
            plan += f"{responses_by_agent['ActivitiesExpert']}\n\n"
        
        plan += "="*70 + "\n"
        plan += " All recommendations are budget-focused as requested!\n"
        plan += "="*70
        
        await ctx.yield_output(plan)

# Create agents
flight_agent, hotel_agent, activities_agent = await create_travel_agents()

# Wrap agents in AgentExecutor
flight_executor = AgentExecutor(flight_agent, id="FlightExpert")
hotel_executor = AgentExecutor(hotel_agent, id="HotelExpert")
activities_executor = AgentExecutor(activities_agent, id="ActivitiesExpert")

# Create custom executors
budget_validator = BudgetValidator()
aggregator = TravelPlanAggregator()

# Build the workflow
workflow = (
    WorkflowBuilder(start_executor=budget_validator)
    .add_fan_out_edges(
        budget_validator,
        [flight_executor, hotel_executor, activities_executor]
    )
    .add_fan_in_edges(
        [flight_executor, hotel_executor, activities_executor],
        aggregator
    )
    .build()
)

# Visualize the workflow using the function
svg_path = visualize_workflow(
    workflow, 
    filename="advanced_multi_agent_workflow.svg",
    show_preview=True
)

In [ ]:
from contextlib import nullcontext

request = "Plan a budget-friendly trip to Barcelona. I need cheap flights, affordable hotels, and free activities."
print(f" User Request:")
print(f"{'─'*70}")
print(f"  {request}")
print(f"{'─'*70}\n")

print(" Executing workflow...\n")
print(" Execution Flow:")
print("="*70)

# Track execution with clean output (no duplicates)
validator_shown = False
step2_shown = False
aggregator_shown = False
agents_completed = set()
final_plan = None

# ★ start_as_current_span() to nest workflow internal spans as children
cm = tracer.start_as_current_span("AdvancedWorkflowExecution") if tracer else nullcontext()
with cm:
    async for event in workflow.run(request, stream=True):
        if isinstance(event, WorkflowEvent):
            if event.type == "executor_invoked":
                if event.executor_id == "budget_validator" and not validator_shown:
                    print(" Step 1: Budget Validation")
                    print("   → Analyzing request for budget keywords...")
                    validator_shown = True
                elif event.executor_id in ["FlightExpert", "HotelExpert", "ActivitiesExpert"]:
                    if not step2_shown:
                        print("\n Step 2: Fan-out to Specialist Agents (Parallel)")
                        step2_shown = True
                elif event.executor_id == "travel_plan_aggregator" and not aggregator_shown:
                    print("\n Step 3: Aggregating Results")
                    print("   → Collecting responses from all agents...")
                    aggregator_shown = True
            
            elif event.type == "executor_completed":
                if event.executor_id not in agents_completed:
                    agents_completed.add(event.executor_id)
                    print(f"   ✓ {event.executor_id}")
            
            elif event.type == "output":
                if isinstance(event.data, list):
                    final_plan = event.data
                elif isinstance(event.data, str):
                    final_plan = event.data
                elif not isinstance(event.data, AgentResponseUpdate):
                    final_plan = event.data

    # Display final plan (once only)
    if final_plan:
        print(f"\n{final_plan}\n")
    else:
        print("\n Final plan was not generated\n")

    print("="*70)
    print(" Workflow Complete!")
    print("="*70)
    print("\n What happened:")
    print("  1, Budget validator detected budget keywords ✓")
    print("  2, Enhanced request with budget context ✓")
    print("  3, Fanned out to 3 agents in parallel ✓")
    print("  4, Each agent provided specialized advice ✓")
    print("  5, Aggregator collected and formatted all responses ✓")
    print("\n Key Benefits of WorkflowBuilder:")
    print("   Custom business logic (budget validation)")
    print("   Parallel execution (fan-out pattern)")
    print("   Result aggregation (fan-in pattern)")
    print("   Full control over workflow structure")
    print("   Deterministic and debuggable execution")


## Validation Point

This pattern (`WorkflowBuilder` + `BudgetValidator` + fan-out/fan-in + `TravelPlanAggregator`) validates **the correctness of the deterministic workflow control infrastructure, not LLM reasoning capabilities**.

---

### 1. Verifying "Pre-processing Gate" Behavior with CustomExecutor

This notebook proves that by simply inheriting `Executor` and defining a `@handler`, you can insert **business logic nodes that don't involve LLMs** into the workflow graph.

Inside the framework, the [`@handler` decorator](jp/use/agent-framework-main-1.0.0b260130/python/packages/core/agent_framework/_workflows/_executor.py) automatically extracts input/output types from the method signature and attaches them as metadata via `_handler_spec`. This means that just the type annotation `validate_budget(self, request: str, ctx: WorkflowContext[str])` in `BudgetValidator` is enough for the framework to recognize it as "a node that receives `str` and sends `str`", guaranteeing type consistency across the graph.

**Validation Point**: Can messages be **transformed and enhanced** via `ctx.send_message()` before flowing downstream?

---

### 2. Structured Parallel Branching with `add_fan_out_edges`

`WorkflowBuilder.add_fan_out_edges` (L533–L612) internally generates a `FanOutEdgeGroup` that **broadcasts the source's message to all targets in parallel**.

```python
.add_fan_out_edges(budget_validator, [flight_executor, hotel_executor, activities_executor])
```

Unlike high-level APIs such as `ConcurrentBuilder`, the key difference is that **any custom Executor can serve as the source**. The official sample `fan_out_fan_in_edges.py` also uses the same structure of `DispatchToExperts(Executor)` → fan-out to 3 agents, demonstrating this as the fundamental pattern.

---

### 3. Type-Safe Aggregation with `add_fan_in_edges`

`add_fan_in_edges` (L830–L920) waits for all sources to complete and passes the results **as `list[T]` to the aggregation** target.

```python
.add_fan_in_edges([flight_executor, hotel_executor, activities_executor], aggregator)
```

The design where `TravelPlanAggregator`'s `@handler` receives `results: list[AgentExecutorResponse]` complies with the framework's **type constraint** (the fan-in target's handler argument must be `list[...]`). Additionally, using `ctx.yield_output()` to emit the **final workflow output** validates the separation of responsibilities from `ctx.send_message()` (which forwards to the next Executor).

---

### 4. Event-Driven Observability

The execution cell retrieves `ExecutorInvokedEvent`, `AgentRunEvent`, and `WorkflowOutputEvent` from `run_stream()`. This is not just logging—it validates that **workflow graph execution is observable and controllable from the outside as an event stream**. Combined with OpenTelemetry tracer integration (with corresponding tests in `test_workflow_observability.py`), this demonstrates a design that enables debugging and monitoring in production environments.

---

### 5. Fundamental Difference from `ConcurrentBuilder`

`ConcurrentBuilder` within the framework performs the same fan-out/fan-in using built-in `_DispatchToAllParticipants` and `_AggregateAgentConversations` as a **high-level API**. The reason this notebook deliberately uses `WorkflowBuilder` is:

| Aspect | ConcurrentBuilder | WorkflowBuilder (this pattern) |
|--------|-------------------|-------------------------------|
| Pre-processing logic | Not possible | Any gate can be inserted via `BudgetValidator` |
| Aggregation logic | Fixed (combine all responses) | Freely formatted via `TravelPlanAggregator` |
| Graph structure | Implicit | Explicit DAG definition + visualization with `WorkflowViz` |
| Extensibility | Agent-only | Any Executor (including non-LLM nodes) |

---

### Summary

What this pattern fundamentally validates is:

> **"Can developers define execution flow as a graph structure without delegating decisions to LLMs, incorporate custom business logic (validation, transformation, aggregation) in a type-safe manner, and build deterministic, observable parallel processing pipelines?"**

## Step 5: AI-Powered Orchestration - MagenticBuilder

**MagenticBuilder** dynamically creates and manages workflows using an AI orchestrator:

### How It Works

1. **You provide**: A set of available agents
2. **AI Orchestrator**: 
   - Analyzes the user's request
   - Creates an execution plan
   - Selects which agents to invoke
   - Determines execution order
   - Adapts the plan based on results

### When to Use MagenticBuilder

 **Use when:**
- Task requirements are complex and unpredictable
- The optimal workflow is not known in advance
- Adaptive planning based on intermediate results is needed
- The task involves open-ended problem solving

 **Don't use when:**
- The workflow is clear and predictable
- Guaranteed execution order is required
- Performance is critical (AI planning adds overhead)
- Simple sequential or parallel patterns are sufficient

### Architecture

```
User Request
     ↓
AI Orchestrator (LLM)
     ↓
Creates Dynamic Plan
     ↓
Executes Agents as Needed
     ↓
Adapts Plan Based on Results
```

In [ ]:
"""
Demonstrates MagenticBuilder: AI-powered dynamic orchestration.

The AI orchestrator:
1. Analyzes the request
2. Creates an execution plan
3. Decides which agents to call and in what order
4. Adapts based on intermediate results
"""
print("="*70)
print("MAGENTIC WORKFLOW: AI-Powered Dynamic Orchestration")
print("="*70)
print("\n🧠 How AI Orchestration Works:")
print("  1. AI orchestrator analyzes the request")
print("  2. Creates a dynamic execution plan")
print("  3. Selects and coordinates agents intelligently")
print("  4. Adapts the plan based on intermediate results")
print("  5. Decides when and in what order to call agents\n")

# Create agents
flight_agent, hotel_agent, activities_agent = await create_travel_agents()

chat_client = AzureOpenAIChatClient(
    **auth_kwargs,
    endpoint=azure_openai_endpoint,
    api_version=api_version,
    deployment_name=azure_deployment,
)

# Create manager agent
manager_agent = chat_client.as_agent(
    instructions="""
    You are a task manager coordinating multiple travel specialists.
    Analyze the user's request, select and delegate to appropriate specialists,
    and synthesize the final travel plan.
    """,
    name="TravelCoordinator",
)

# Build magentic workflow - AI orchestrator coordinates everything
workflow = (
    MagenticBuilder(
        participants=[flight_agent, hotel_agent, activities_agent],
        manager_agent=manager_agent,
        max_round_count=10,
        max_stall_count=3,
        max_reset_count=2,
    )
    .build()
)


# Visualize the workflow using the function
svg_path = visualize_workflow(
    workflow, 
    filename="magentic_multi_agent_workflow.svg",
    show_preview=True
)

In [ ]:

# The orchestrator decides which agents to call and in what order
request = (
    "I need to plan a 4-day romantic anniversary trip to Venice. "
    "I want a nice hotel near San Marco, romantic restaurants, and gondola rides. "
    "What should I do?"
)

print(f" User Request:")
print(f"{'─'*70}")
print(f"  {request}")
print(f"{'─'*70}\n")

print(" Starting AI Orchestration...\n")
print(" Execution Flow:")
print("="*70)

# Track what happens
executor_invoked = []
agent_runs = []
final_output = None

# ★ start_as_current_span() to nest workflow internal spans as children
cm = tracer.start_as_current_span("MagenticWorkflowExecution") if tracer else nullcontext()
with cm:
    # Run the workflow and track events
    async for event in workflow.run(request, stream=True):
        if isinstance(event, WorkflowEvent):
            if event.type == "executor_invoked":
                executor_invoked.append(event.executor_id)
                # Show orchestrator activity
                if "orchestrator" in event.executor_id.lower():
                    print("   🧠 AI Orchestrator: Creating execution plan...")
                else:
                    agent_name = event.executor_id.replace('agent_', '').replace('_expert', ' Expert')
                    print(f"   🤖 Delegating to: {agent_name}")
            
            elif event.type == "executor_completed":
                if event.executor_id and event.executor_id not in agent_runs:
                    agent_runs.append(event.executor_id)
                    agent_name = event.executor_id.replace('agent_', '').replace('_expert', ' Expert')
                    print(f"   ✓ {agent_name} completed")
            
            elif event.type == "output":
                if isinstance(event.data, list):
                    final_output = event.data
                elif not isinstance(event.data, AgentResponseUpdate):
                    final_output = event.data

    # Show what happened
    print("\n" + "="*70)
    print(" Orchestration Summary")
    print("="*70 + "\n")

    print(f"🧠 Orchestration Details:")
    print(f"   • Executors invoked: {len(executor_invoked)}")
    print(f"   • Agents delegated: {len([e for e in executor_invoked if 'expert' in e.lower()])}")
    print(f"   • Orchestrator planning rounds: {len([e for e in executor_invoked if 'orchestrator' in e.lower()])}")

    agents_called = [e for e in executor_invoked if 'expert' in e.lower()]
    if agents_called:
        print(f"\n🤖 Agents Called (in order):")
        for agent_id in agents_called:
            clean_name = agent_id.replace('agent_', '').replace('_expert', ' Expert')
            print(f"   • {clean_name}")

    # Display final result
    if final_output:
        print("\n" + "="*70)
        print(" Final Orchestration Result")
        print("="*70 + "\n")
        
        # Extract text from Message
        result_text = None
        if hasattr(final_output, 'text'):
            result_text = final_output.text
        elif isinstance(final_output, str):
            result_text = final_output
        elif isinstance(final_output, list):
            # Might be a list of messages
            texts = []
            for item in final_output:
                if hasattr(item, 'text'):
                    texts.append(item.text)
                elif isinstance(item, str):
                    texts.append(item)
            result_text = " ".join(texts) if texts else None
        
        if result_text:
            # Format and wrap text
            words = result_text.split()
            line = ""
            for word in words:
                if len(line + word) > 66:
                    print(f"  {line}")
                    line = word + " "
                else:
                    line += word + " "
            if line:
                print(f"  {line.strip()}")
        else:
            print(f"  Type: {type(final_output)}")
            print(f"  Data: {final_output}")
        print()
    else:
        print("\n Final output was not captured\n")

    print("="*70)
    print(" MAGENTIC WORKFLOW COMPLETE!")
    print("="*70)
    print("\n What happened:")
    print(f"  1, AI orchestrator analyzed the romantic Venice request")
    print(f"  2, Dynamically chose to call {len(agents_called)} agents")
    print(f"  3, Created {len([e for e in executor_invoked if 'orchestrator' in e.lower()])} planning rounds")
    print(f"  4, Synthesized the final travel plan")
    print(f"  5, No coordination logic needed to be coded!")
    print("\n Key Benefits of MagenticBuilder:")
    print("   AI dynamically creates the optimal execution plan")
    print("   Automatically adapts to request complexity")
    print("   No hardcoded workflow logic needed")
    print("   Easy to scale - just add specialist agents")
    print("   Handles unpredictable and complex requests gracefully")


## Validating the Viability of "Delegating Orchestration Itself to LLMs"

While the first pattern tested "deterministic control where developers explicitly define the DAG", MagenticBuilder validates whether **"non-deterministic orchestration where the LLM dynamically decides which agents to call, in what order, and how many times"** functions at a practical level.

---

### 1. Internal Architecture: Automatic Conversion to Star Graph

Inside `MagenticBuilder.build()`, it **actually uses WorkflowBuilder**:

```python
def build(self) -> Workflow:
    participants = self._resolve_participants()
    orchestrator = self._resolve_orchestrator(participants)
    
    workflow_builder = WorkflowBuilder().set_start_executor(orchestrator)
    for participant in participants:
        workflow_builder = workflow_builder.add_edge(orchestrator, participant)
        workflow_builder = workflow_builder.add_edge(participant, orchestrator)
    return workflow_builder.build()
```

So the internal graph is **bidirectional edges between the orchestrator and all agents** (star topology):

```
         ┌──── FlightExpert
         │
Orchestrator ──── HotelExpert
         │
         └──── ActivitiesExpert
         (all bidirectional)
```

This is fundamentally different from the first pattern's unidirectional DAG (`BudgetValidator → fan-out → fan-in → Aggregator`), and **which path is taken is determined by the LLM's judgment, not by the graph structure**.

---

### 2. Task Ledger: Fact Organization and Planning Before Execution

**Before** entering the orchestration loop, the manager LLM generates a **`_MagenticTaskLedger`**. This is a two-phase LLM call performed in the `plan()` method (`_magentic.py` L610–L637):

**Phase 1: Facts (Organizing facts)** — Sends `ORCHESTRATOR_TASK_LEDGER_FACTS_PROMPT` to the LLM, categorizing task facts into 4 categories:

```
1. GIVEN OR VERIFIED FACTS   — Facts explicitly stated in the request
2. FACTS TO LOOK UP          — Facts that need to be researched and their sources
3. FACTS TO DERIVE           — Facts to be derived through reasoning/calculation
4. EDUCATED GUESSES           — Facts based on memory/estimation
```

**Phase 2: Plan (Formulating the plan)** — Uses `ORCHESTRATOR_TASK_LEDGER_PLAN_PROMPT` to generate a bulleted execution plan based on team composition and facts.

These two are saved as `_MagenticTaskLedger(facts=facts_msg, plan=plan_msg)`, integrated into the orchestrator's context via `ORCHESTRATOR_TASK_LEDGER_FULL_PROMPT`.

**Update on Reset**: During `replan()` after stall detection, the Task Ledger is regenerated using `ORCHESTRATOR_TASK_LEDGER_FACTS_UPDATE_PROMPT` (reflecting "what new information was learned") and `ORCHESTRATOR_TASK_LEDGER_PLAN_UPDATE_PROMPT` (a new plan considering "what failed last time").

**Validation Point**: Can the LLM be forced into a cognitive process of **structured fact organization → planning → round execution based on the plan → re-planning on failure**, rather than "calling agents immediately"?

---

### 3. Progress Ledger: LLM Control Decisions Per Round

While the Task Ledger is the "pre-execution plan", the Progress Ledger handles **control decisions at each round**. `create_progress_ledger()` is called within the `MagenticOrchestrator` loop, and the manager LLM outputs a **`MagenticProgressLedger`** as a JSON structure:

```python
# Progress Ledger fields:
{
    "is_request_satisfied": bool,    # Is the task complete?
    "is_in_loop": bool,              # Are we stuck in a loop?
    "is_progress_being_made": bool,  # Is progress being made?
    "next_speaker": str,             # Which agent to call next
    "instruction_or_question": str   # Instructions for that agent
}
```

**Relationship between the two Ledgers**: Since the Task Ledger's facts/plan are included in `chat_history` when the Progress Ledger is generated, the Progress Ledger **references the Task Ledger's plan while determining "the next move"**.

**Validation Point**: Can the LLM make a structured judgment of "who to call next" at every round, and can this dynamically determine the workflow's execution flow? In contrast to the first pattern's static routing with `add_fan_out_edges`, **routing itself is the output of LLM reasoning**.

---

### 4. Safety Valves: Stall Detection and Reset Mechanism

The three parameters set in the notebook are **guardrails against LLM non-determinism**:

| Parameter | Notebook Value | What It Controls |
|-----------|---------------|------------------|
| `max_round_count=10` | Upper limit on total orchestration rounds. Forces termination on exceed |
| `max_stall_count=3` | Allowed consecutive rounds without progress. Triggers reset & re-planning on exceed |
| `max_reset_count=2` | Maximum number of resets (re-plans). Forces termination on exceed |

The internal flow is:

```
progress_being_made=False → stall_count++
stall_count > max_stall_count → reset all participants + re-plan
reset_count >= max_reset_count → force terminate workflow
round_index >= max_round_count → force terminate workflow
```

`MagenticAgentExecutor` has a `handle_magentic_reset()` handler that clears all caches, conversation history, and threads on reset. This is a **fault-tolerance design specific to LLM orchestration** that doesn't exist in the first pattern.

---

### 5. What the Execution Cell's Event Tracking Validates

The notebook tracks `ExecutorInvokedEvent` / `AgentRunEvent` to visualize:

- Number of `orchestrator` invocations → **How many times the LLM "thought"**
- Order of `expert` invocations → **In what order the LLM chose agents**

In the first pattern, these were **predictable from the graph definition**, but here they are **unknown until execution**. The fact that the same request can result in different agent call orders across runs is itself the validation target.

---

### 6. Fundamental Contrast Between the Two Patterns

| Aspect | Pattern 1 (WorkflowBuilder) | Pattern 2 (MagenticBuilder) |
|--------|---------------------------|---------------------------|
| **"Who to call" decision** | Developer defines in graph | LLM judges each round |
| **Execution order** | Deterministic & reproducible | Non-deterministic & adaptive |
| **Pre/post processing** | Freely implemented in CustomExecutor | Auto-integrated by orchestrator |
| **Parallel execution** | Explicit parallelization via fan-out | LLM calls agents sequentially one at a time |
| **Error recovery** | Developer implements | Built-in stall detection → reset → re-plan |
| **Internal implementation** | Developer-defined DAG | `build()` auto-generates star graph |
| **Adding agents** | Requires graph redefinition | Just add to `.participants()` |
| **Runtime overhead** | Low | High (AI planning) |
| **Best for** | Known workflows | Unknown/complex tasks |
| **Debugging** | Easy (explicit graph) | Difficult (AI decisions) |
| **Scalability** | Manual updates needed | Auto-adapts to new agents |

---

### Summary

> **"Can practical multi-agent collaboration be achieved for complex tasks where flow cannot be pre-defined, by delegating workflow control flow itself to LLMs and combining Task Ledger-based pre-execution fact organization and planning + Progress Ledger-based structured per-round decisions + stall detection/reset guardrails?"**

While the first pattern's design was "containing AI non-determinism within deterministic workflow control", the second conversely **"leverages AI non-determinism in workflow control itself while preventing runaway behavior with safety valves (max_round/stall/reset)"**. By placing these two in contrast, the overall tutorial demonstrates the design tradeoff of **where to place control—with the developer or with AI**.

##  Comparing Advanced Patterns

### WorkflowBuilder vs MagenticBuilder

### Real-World Use Cases

**WorkflowBuilder Examples:**
```python
# Approval workflow
workflow = (
    WorkflowBuilder()
    .set_start_executor(request_handler)
    .add_edge(request_handler, manager_approval)
    .add_switch_case_edge_group(
        manager_approval,
        [
            Case(lambda x: x.approved, finance_review),
            Default(rejection_handler)
        ]
    )
    .build()
)

# Multi-stage pipeline
workflow = (
    WorkflowBuilder()
    .add_chain([data_validator, processor, quality_check, publisher])
    .build()
)

# Fan-out/fan-in pattern
workflow = (
    WorkflowBuilder()
    .set_start_executor(splitter)
    .add_fan_out_edges(splitter, [worker1, worker2, worker3])
    .add_fan_in_edges([worker1, worker2, worker3], aggregator)
    .build()
)
```

**MagenticBuilder Examples:**
```python
# Research assistant
workflow = (
    MagenticBuilder()
    .participants([web_researcher, academic_searcher, data_analyst, summarizer])
    .with_manager(
        agent=manager_agent,
        max_round_count=10
    )
    .build()
)

# Customer support
workflow = (
    MagenticBuilder()
    .participants([product_expert, billing_expert, tech_support, escalation_handler])
    .with_manager(
        agent=manager_agent,
        max_round_count=8
    )
    .build()
)
```

### Key Differences

**WorkflowBuilder:**
-  Defines the entire execution graph
-  Predictable, testable, reproducible
-  Suitable for compliance and auditing
-  Requires upfront design
-  Inflexible structure

**MagenticBuilder:**
-  AI orchestrator creates the execution plan
-  Adapts to complex and diverse requests
-  Easy to add new specialist agents
-  Less predictable
-  Difficult to debug AI decisions

##  Key Takeaways

### What You Learned

1. **CustomExecutor**
   - Inherit `Executor` and call `super().__init__(id="...")`
   - Use `@handler` with `WorkflowContext` type annotation
   - Implement business logic before passing to agents
   - Examples: validation, transformation, routing

2. **WorkflowBuilder**
   - Full control over workflow graph structure
   - Supports fan-out, fan-in, conditional routing
   - Best for well-defined complex workflows
   - Deterministic and debuggable

3. **MagenticBuilder**
   - AI orchestrator dynamically manages execution
   - Best for unpredictable, adaptive workflows
   - Easily scales to many specialist agents
   - Higher overhead but more flexible

4. **Choosing the Right Pattern**
   - **Known workflows** → WorkflowBuilder
   - **Unknown/adaptive** → MagenticBuilder
   - **Simple sequential** → SequentialBuilder (Tutorial 05)
   - **Simple parallel** → ConcurrentBuilder (Tutorial 05)

### Production Patterns

```python
# CustomExecutor template
class MyExecutor(Executor):
    def __init__(self):
        super().__init__(id="my_executor")
    
    @handler
    async def process(self, input: str, ctx: WorkflowContext[str]):
        # Business logic here
        result = do_something(input)
        await ctx.send_message(result)

# WorkflowBuilder with validation gate
workflow = (
    WorkflowBuilder()
    .set_start_executor(validator)
    .add_fan_out_edges(validator, [agent1, agent2, agent3])
    .build()
)

# MagenticBuilder for adaptive tasks
workflow = (
    MagenticBuilder()
    .participants([specialist1, specialist2, specialist3])
    .with_manager(agent=manager_agent, max_round_count=10)
    .build()
)
```

##  Exercises

### Exercise 1: Multi-Stage Approval Workflow

Create a workflow with conditional routing:
```
Request → Validation → (if valid) → Manager Approval
                                    ├─→ (if approved) → Finance
                                    └─→ (if rejected) → Rejection Handler
```

**Hint:** Use `add_switch_case_edge_group()` for conditional routing.

### Exercise 2: Data Processing Pipeline

Create a fan-out/fan-in pattern:
```
Data Splitter → Worker 1 ↘
             → Worker 2  → Aggregation → Final Processor
             → Worker 3 ↗
```

**Hint:** Use `add_fan_out_edges()` and `add_fan_in_edges()`.

### Exercise 3: Adaptive Research System

Create a magentic workflow with:
- Web researcher
- Academic searcher
- Data analyst
- Fact checker
- Summarizer

Let the AI orchestrator decide how to coordinate them!

In [ ]:
# Exercise Playground - Implement your solutions here!

async def approval_workflow_exercise():
    """
    Exercise 1: Build a multi-stage approval workflow.
    """
    # TODO: Implement approval workflow with conditional routing
    pass

async def pipeline_exercise():
    """
    Exercise 2: Build a fan-out/fan-in data processing pipeline.
    """
    # TODO: Implement data processing pipeline
    pass

async def research_system_exercise():
    """
    Exercise 3: Build an adaptive research system.
    """
    # TODO: Implement magentic research workflow
    pass

print(" Exercise templates ready - implement your solutions above!")

##  Next Steps

Congratulations! You've mastered advanced multi-agent workflow patterns!

You can now:
-  Build CustomExecutors with business logic
-  Create complex workflow graphs with WorkflowBuilder
-  Implement fan-out/fan-in patterns
-  Use AI-powered orchestration with MagenticBuilder
-  Choose the right pattern for your use case


---

### Quick Reference

**CustomExecutor:**
```python
class MyExecutor(Executor):
    def __init__(self):
        super().__init__(id="my_id")
    
    @handler
    async def process(self, msg: str, ctx: WorkflowContext[str]):
        result = transform(msg)
        await ctx.send_message(result)
```

**WorkflowBuilder:**
```python
workflow = (
    WorkflowBuilder()
    .set_start_executor(start)
    .add_fan_out_edges(start, [a, b, c])
    .add_edge(a, end)
    .build()
)
```

**MagenticBuilder:**
```python
workflow = (
    MagenticBuilder()
    .participants([agent1, agent2, agent3])
    .with_manager(agent=manager_agent, max_round_count=10)
    .build()
)
```

**Running a workflow:**
```python
# Streaming
async for event in workflow.run_stream(input):
    if isinstance(event, AgentRunEvent):
        print(event.data)

# Non-streaming
result = await workflow.run(input)
outputs = result.get_outputs()
```